# Generate SATURN ESM2 gene embeddings for nine species

This notebook creates SATURN-ready PyTorch `.pt` files from the local `esm2_t36_3B_UR50D` checkpoint. It uses **representation layer 33** and mean-pools amino-acid residues, matching the requested layer rather than using the last layer.

All new artifacts are written below `orthofind/proteomes/saturn_esm2_t36_3B_layer33/`. Existing TranscriptFormer HDF5 files and its normalized FASTA/mapping files are never read, modified, or overwritten.


In [ ]:
# Cell 1. Imports
from pathlib import Path
import csv
import time
from urllib.parse import quote, unquote

import numpy as np
import torch
from Bio import SeqIO
from esm import FastaBatchedDataset
from esm.pretrained import load_model_and_alphabet_core
from tqdm.auto import tqdm


In [ ]:
# Cell 2. Paths, species configuration, and ESM2 settings
ESM_MODEL = Path("./models/esm2_t36_3B_UR50D.pt")
MODEL_NAME = "esm2_t36_3B_UR50D"
REPR_LAYER = 33
MAX_RESIDUES = 1022
SEQUENCES_PER_BATCH_TARGET = 1
TOKENS_PER_BATCH = MAX_RESIDUES * SEQUENCES_PER_BATCH_TARGET
USE_FP16_ON_GPU = False
OUTPUT_DTYPE = np.float32

# False means a valid SATURN .pt is preserved and that species is skipped.
# This affects only SATURN_OUTPUT_ROOT, never TranscriptFormer files.
OVERWRITE_EXISTING = False
CLEAR_CUDA_CACHE_BETWEEN_SPECIES = True

BASE_DIR = Path("./orthofind/proteomes")
SATURN_OUTPUT_ROOT = BASE_DIR / "saturn_esm2_t36_3B_layer33"

SPECIES_CONFIG = {
    "crocodylus_siamensis": ("Siamese crocodile", "cro.fa"),
    "dromaius_novaehollandiae": ("Emu", "emu.fa"),
    "elaphe_schrenckii": ("Elaphe schrenckii", "esc.fa"),
    "gallus_gallus": ("Chicken", "gga.fa"),
    "homo_sapiens": ("Human", "hsa.fa"),
    "mus_musculus": ("Mouse", "mmu.fa"),
    "pelodiscus_sinensis": ("Chinese softshell turtle", "psi.fa"),
    "pogona_vitticeps": ("Central bearded dragon", "pvi.fa"),
    "trachemys_scripta_elegans": ("Red-eared slider", "tse.fa"),
}

SPECIES_CONFIG = {
    species: {
        "common_name": common_name,
        "fasta": BASE_DIR / fasta_name,
        # Only SATURN-specific normalized inputs/mappings are written here.
        "work_dir": SATURN_OUTPUT_ROOT / species,
        "normalized_fasta": SATURN_OUTPUT_ROOT / species / f"{species}.esm2_input.fa",
        "id_mapping_tsv": SATURN_OUTPUT_ROOT / species / f"{species}.gene_id_mapping.tsv",
        "output_pt": SATURN_OUTPUT_ROOT / species / f"{species}_gene_embeddings_ESM2_t36_3B_layer33.pt",
    }
    for species, (common_name, fasta_name) in SPECIES_CONFIG.items()
}

assert ESM_MODEL.is_file(), f"Cannot find ESM2 model: {ESM_MODEL}"
for species, config in SPECIES_CONFIG.items():
    assert config["fasta"].is_file(), f"[{species}] missing FASTA: {config['fasta']}"

print("Model:", ESM_MODEL.resolve())
print("Representation layer:", REPR_LAYER)
print("SATURN output root:", SATURN_OUTPUT_ROOT)
print("Species:", ", ".join(SPECIES_CONFIG))


In [ ]:
# Cell 3. ID handling and protein cleaning
# The first whitespace-delimited FASTA header field is treated as the gene ID,
# exactly as in the existing TranscriptFormer preparation workflow.
def get_raw_gene_id(record):
    return record.id


def make_safe_id(raw_gene_id):
    raw_gene_id = str(raw_gene_id).strip()
    if not raw_gene_id or "\x00" in raw_gene_id:
        raise ValueError(f"Invalid gene ID: {raw_gene_id!r}")
    # A slash must not become an HDF5/path component in downstream tooling.
    return quote(raw_gene_id, safe="")


def restore_raw_gene_id(safe_gene_id):
    return unquote(safe_gene_id)


VALID_AMINO_ACIDS = set("ACDEFGHIKLMNPQRSTVWYBXZOU")


def clean_protein_sequence(sequence):
    sequence = str(sequence).upper().replace("*", "")
    return "".join(aa if aa in VALID_AMINO_ACIDS else "X" for aa in sequence)


assert make_safe_id("PGR2/3") == "PGR2%2F3"
assert restore_raw_gene_id("PGR2%2F3") == "PGR2/3"


In [ ]:
# Cell 4. Create SATURN-only normalized FASTA files and ID mappings
def prepare_normalized_fasta(species, config):
    normalized_records = []
    mapping_rows = []
    seen_gene_ids = set()
    total = empty = duplicates = long_sequences = 0

    for record in tqdm(SeqIO.parse(config["fasta"], "fasta"), desc=f"Normalize {species}"):
        total += 1
        raw_gene_id = str(get_raw_gene_id(record)).strip()
        safe_gene_id = make_safe_id(raw_gene_id)
        sequence = clean_protein_sequence(record.seq)
        if not sequence:
            empty += 1
            continue
        # Keep the first nonempty protein per gene, matching the supplied
        # TranscriptFormer workflow. This produces one embedding per raw gene ID.
        if safe_gene_id in seen_gene_ids:
            duplicates += 1
            continue
        seen_gene_ids.add(safe_gene_id)
        long_sequences += len(sequence) > MAX_RESIDUES
        normalized_records.append(
            record.__class__(record.seq.__class__(sequence), id=safe_gene_id, name=safe_gene_id, description="")
        )
        mapping_rows.append((raw_gene_id, safe_gene_id))

    if not normalized_records:
        raise ValueError(f"[{species}] no usable protein sequences")
    config["work_dir"].mkdir(parents=True, exist_ok=True)
    SeqIO.write(normalized_records, config["normalized_fasta"], "fasta")
    with config["id_mapping_tsv"].open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle, delimiter="\t")
        writer.writerow(("raw_gene_id", "safe_gene_id"))
        writer.writerows(mapping_rows)
    print(f"[{species}] kept {len(normalized_records):,}/{total:,}; empty={empty:,}; duplicates={duplicates:,}; >{MAX_RESIDUES} aa={long_sequences:,}")


for species, config in SPECIES_CONFIG.items():
    if config["output_pt"].is_file() and not OVERWRITE_EXISTING:
        print(f"[{species}] valid output will be checked before processing")
    else:
        prepare_normalized_fasta(species, config)


In [ ]:
# Cell 5. Load the local ESM2 3B model once
assert ESM_MODEL.stem == MODEL_NAME
model_data = torch.load(str(ESM_MODEL), map_location="cpu", weights_only=False)
model, alphabet = load_model_and_alphabet_core(
    model_name=MODEL_NAME,
    model_data=model_data,
    regression_data=None,
)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    if USE_FP16_ON_GPU:
        model = model.half()
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA not detected; ESM2 3B on CPU will be very slow.")

assert model.num_layers == 36
assert model.embed_dim == 2560
assert 0 <= REPR_LAYER <= model.num_layers
batch_converter = alphabet.get_batch_converter(truncation_seq_length=MAX_RESIDUES)
print(f"Loaded {MODEL_NAME}: layer={REPR_LAYER}, dim={model.embed_dim}, device={device}")


In [ ]:
# Cell 6. ESM2 inference and SATURN .pt writer
def output_pt_is_valid(path):
    if not path.is_file():
        return False
    try:
        embeddings = torch.load(path, map_location="cpu", weights_only=False)
        return (
            isinstance(embeddings, dict)
            and bool(embeddings)
            and all(
                isinstance(gene_id, str)
                and isinstance(embedding, torch.Tensor)
                and embedding.dtype == torch.float32
                and embedding.device.type == "cpu"
                and tuple(embedding.shape) == (2560,)
                for gene_id, embedding in embeddings.items()
            )
        )
    except (OSError, RuntimeError, ValueError, KeyError):
        return False


def process_one_species(species, config):
    output_pt = config["output_pt"]
    if output_pt_is_valid(output_pt) and not OVERWRITE_EXISTING:
        print(f"[{species}] existing valid SATURN .pt; skipped: {output_pt}")
        return
    # Cell 4 may have been skipped after changing OVERWRITE_EXISTING in-place.
    if not config["normalized_fasta"].is_file():
        prepare_normalized_fasta(species, config)

    dataset = FastaBatchedDataset.from_file(str(config["normalized_fasta"]))
    batches = dataset.get_batch_indices(toks_per_batch=TOKENS_PER_BATCH, extra_toks_per_seq=1)
    data_loader = torch.utils.data.DataLoader(
        dataset,
        collate_fn=batch_converter,
        batch_sampler=batches,
        num_workers=0,
        pin_memory=device.type == "cuda",
    )
    embeddings = {}
    started = time.perf_counter()
    with torch.inference_mode():
        for labels, sequences, tokens in tqdm(data_loader, total=len(batches), desc=f"ESM2 {species}"):
            tokens = tokens.to(device, non_blocking=device.type == "cuda")
            representations = model(tokens, repr_layers=[REPR_LAYER], return_contacts=False)["representations"][REPR_LAYER]
            for index, (safe_gene_id, sequence) in enumerate(zip(labels, sequences)):
                effective_length = min(len(sequence), MAX_RESIDUES)
                embedding = representations[index, 1:effective_length + 1].float().mean(dim=0).cpu().to(torch.float32)
                embeddings[restore_raw_gene_id(safe_gene_id)] = embedding

    if len(embeddings) != len(dataset):
        raise RuntimeError(f"[{species}] expected {len(dataset)} embeddings, got {len(embeddings)}")
    if not all(tuple(embedding.shape) == (2560,) for embedding in embeddings.values()):
        raise RuntimeError(f"[{species}] an embedding does not have shape (2560,)")
    # Atomic replacement prevents a partially written file appearing complete.
    temporary_pt = output_pt.with_name(f"{output_pt.name}.tmp")
    torch.save(embeddings, temporary_pt)
    temporary_pt.replace(output_pt)
    print(f"[{species}] wrote {len(embeddings):,} genes in {(time.perf_counter() - started) / 60:.2f} min: {output_pt}")
    if device.type == "cuda" and CLEAR_CUDA_CACHE_BETWEEN_SPECIES:
        torch.cuda.empty_cache()


In [ ]:
# Cell 7. Run all nine species
for species, config in SPECIES_CONFIG.items():
    process_one_species(species, config)

print("All SATURN ESM2 layer-33 files are complete.")


In [ ]:
# Cell 8. Verify the final SATURN files
for species, config in SPECIES_CONFIG.items():
    path = config["output_pt"]
    assert output_pt_is_valid(path), f"Invalid SATURN output: {path}"
    embeddings = torch.load(path, map_location="cpu", weights_only=False)
    first_gene_id, first_embedding = next(iter(embeddings.items()))
    print(f"{species}: genes={len(embeddings):,}; {first_gene_id!r}; shape={tuple(first_embedding.shape)}; dtype={first_embedding.dtype}; device={first_embedding.device}")
